# Production Trade Report

Pull historical trade data directly from the local MetaTrader 5 terminal and summarise production performance by symbol.

**Workflow**
1. Set the date range and optional symbol/group filters
2. Run the fetch cell to load MT5 deal history
3. Review the per-symbol summary, daily PnL, and best/worst trades


In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from Learn.report import fetch_trade_report

In [2]:
# -- Configuration -----------------------------------------------------------
START_DATE = "2026-04-19"
END_DATE   = pd.Timestamp.now("UTC").strftime("%Y-%m-%d")

GROUP       = "*"      # MT5 group filter, e.g. "*USD*" or "*"
SYMBOLS     = None     # e.g. ["EURUSD.a", "US500.a"]
CLOSED_ONLY = True     # Summary focuses on close-side deals only


In [3]:
trades, summary = fetch_trade_report(
    start_date=START_DATE,
    end_date=END_DATE,
    group=GROUP,
    symbols=SYMBOLS,
    closed_only=CLOSED_ONLY,
)

closed_trades = trades[trades["entry_type"].isin(["OUT", "OUT_BY", "INOUT"])].copy()
closed_trades["trade_date"] = closed_trades["time"].dt.tz_convert(None).dt.floor("D") if not closed_trades.empty else pd.Series(dtype="datetime64[ns]")


In [4]:
print(f"Rows returned      : {len(trades):,}")
print(f"Closed-trade rows  : {len(closed_trades):,}")
print(f"Symbols returned   : {sorted(trades['symbol'].dropna().unique().tolist()) if not trades.empty else []}")
if not trades.empty:
    print(f"Time range (UTC)   : {trades['time'].min()} -> {trades['time'].max()}")

display(trades.head(10))

Rows returned      : 184
Closed-trade rows  : 92
Symbols returned   : ['EURUSD.a', 'US500.a', 'XAUUSD.a']
Time range (UTC)   : 2026-04-20 04:59:14+00:00 -> 2026-04-24 21:22:56+00:00


,ticket,order,position_id,time,time_msc,symbol,side,deal_type,entry_type,reason,volume,price,profit,commission,swap,fee,net_pnl,magic,comment,external_id
0,230530595,289625914,289625914,2026-04-20 04:59:14+00:00,2026-04-20 04:59:14.459000+00:00,EURUSD.a,buy,BUY,IN,EXPERT,1.25,1.17576,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
1,230539380,289639884,289625914,2026-04-20 05:30:01+00:00,2026-04-20 05:30:01.090000+00:00,EURUSD.a,sell,SELL,OUT,TP,1.25,1.17606,52.41,-4.38,0.0,0.0,48.03,235000,[tp 1.17606],
2,230760881,289947428,289947428,2026-04-20 15:23:12+00:00,2026-04-20 15:23:12.598000+00:00,EURUSD.a,sell,SELL,IN,EXPERT,1.25,1.17602,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
3,230772630,289961383,289947428,2026-04-20 15:41:57+00:00,2026-04-20 15:41:57.570000+00:00,EURUSD.a,buy,BUY,OUT,SL,1.25,1.17646,-76.84,-4.38,0.0,0.0,-81.22,235000,[sl 1.17646],
4,230938194,290155348,290155348,2026-04-20 18:54:12+00:00,2026-04-20 18:54:12.900000+00:00,EURUSD.a,buy,BUY,IN,EXPERT,1.25,1.17835,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
5,230950043,290170311,290155348,2026-04-20 19:19:07+00:00,2026-04-20 19:19:07.585000+00:00,EURUSD.a,sell,SELL,OUT,TP,1.25,1.17881,80.14,-4.38,0.0,0.0,75.76,235000,[tp 1.17881],
6,230990855,290227844,290227844,2026-04-20 21:28:00+00:00,2026-04-20 21:28:00.855000+00:00,EURUSD.a,buy,BUY,IN,EXPERT,1.25,1.17873,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
7,230991010,290228358,290227844,2026-04-20 21:28:39+00:00,2026-04-20 21:28:39.383000+00:00,EURUSD.a,sell,SELL,OUT,SL,1.25,1.17841,-55.74,-4.38,0.0,0.0,-60.12,235000,[sl 1.17841],
8,231118541,290422193,290422193,2026-04-21 06:54:09+00:00,2026-04-21 06:54:09.403000+00:00,XAUUSD.a,sell,SELL,IN,EXPERT,0.10,4796.45000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,
9,231120367,290424929,290424929,2026-04-21 07:02:24+00:00,2026-04-21 07:02:24.425000+00:00,XAUUSD.a,sell,SELL,IN,EXPERT,0.10,4797.02000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,


## Per-symbol summary

In [5]:
if summary.empty:
    print("No symbol summary available for the selected range/filter.")
else:
    summary_display = summary.copy()
    summary_display["win_rate"] = (summary_display["win_rate"] * 100).round(2)
    display(summary_display.round({
        "volume_lots": 2,
        "gross_profit": 2,
        "gross_loss": 2,
        "net_pnl": 2,
        "avg_net_pnl": 2,
        "median_net_pnl": 2,
        "win_rate": 2,
        "avg_win": 2,
        "avg_loss": 2,
        "total_commission": 2,
        "total_swap": 2,
        "total_fee": 2,
    }))

,symbol,trade_count,first_trade_time,last_trade_time,volume_lots,gross_profit,gross_loss,net_pnl,avg_net_pnl,median_net_pnl,win_rate,avg_win,avg_loss,total_commission,total_swap,total_fee
0,EURUSD.a,21,2026-04-20 05:30:01+00:00,2026-04-24 20:00:07+00:00,26.25,1070.39,-563.29,507.10,24.15,65.70,66.67,76.46,-80.47,-91.98,0.0,0.0
1,US500.a,23,2026-04-21 15:38:31+00:00,2026-04-24 21:22:56+00:00,115.00,300.67,-346.64,-45.97,-2.00,9.78,52.17,25.06,-31.51,0.00,0.0,0.0
2,XAUUSD.a,48,2026-04-21 07:14:25+00:00,2026-04-24 18:34:56+00:00,4.80,1710.41,-2114.43,-404.02,-8.42,-1.32,50.00,71.27,-88.10,0.00,0.0,0.0


## Daily net PnL and cumulative PnL

In [6]:
if closed_trades.empty:
    print("No closed trades available to chart.")
else:
    daily = (
        closed_trades.groupby(["trade_date", "symbol"], as_index=False)["net_pnl"]
        .sum()
        .sort_values(["trade_date", "symbol"])
    )
    daily_pivot = daily.pivot(index="trade_date", columns="symbol", values="net_pnl").fillna(0.0)
    cumulative = daily_pivot.cumsum()

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.10,
        subplot_titles=("Daily net PnL by symbol", "Cumulative net PnL by symbol"),
    )

    for symbol in daily_pivot.columns:
        fig.add_trace(
            go.Scatter(
                x=daily_pivot.index,
                y=daily_pivot[symbol],
                mode="lines+markers",
                name=f"{symbol} daily",
                legendgroup=str(symbol),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=cumulative.index,
                y=cumulative[symbol],
                mode="lines",
                name=f"{symbol} cumulative",
                legendgroup=str(symbol),
                showlegend=False,
            ),
            row=2,
            col=1,
        )

    zero_line_style = dict(color="black", width=1, dash="dash")
    fig.add_hline(y=0, line=zero_line_style, row=1, col=1)
    fig.add_hline(y=0, line=zero_line_style, row=2, col=1)
    fig.update_yaxes(title_text="Net PnL", row=1, col=1)
    fig.update_yaxes(title_text="Cumulative net PnL", row=2, col=1)
    fig.update_xaxes(title_text="Trade date", row=2, col=1)
    fig.update_layout(height=850, hovermode="x unified", template="plotly_white")
    fig.show()

    display(daily.tail(20))

,trade_date,symbol,net_pnl
0,2026-04-20,EURUSD.a,-17.55
1,2026-04-21,EURUSD.a,358.55
2,2026-04-21,US500.a,-9.67
3,2026-04-21,XAUUSD.a,-34.65
4,2026-04-22,EURUSD.a,-227.88
5,2026-04-22,US500.a,-67.67
6,2026-04-22,XAUUSD.a,310.55
7,2026-04-23,EURUSD.a,185.11
8,2026-04-23,US500.a,35.56
9,2026-04-23,XAUUSD.a,-183.38


In [7]:
if closed_trades.empty:
    print("No closed trades available to chart.")
else:
    plot_df = closed_trades.copy()
    plot_df["result"] = plot_df["net_pnl"].apply(lambda x: "win" if x > 0 else ("loss" if x < 0 else "breakeven"))
    plot_df["abs_profit"] = plot_df["net_pnl"].abs()

    # Remove top 5% outliers per symbol
    # p95_by_symbol = plot_df.groupby("symbol")["abs_profit"].transform(lambda s: s.quantile(0.95))
    # plot_df = plot_df[plot_df["abs_profit"] < p95_by_symbol]

    symbols = sorted(plot_df["symbol"].dropna().unique().tolist())
    if not symbols:
        print("No data left after filtering.")
    else:
        n_cols = 2
        n_rows = (len(symbols) + n_cols - 1) // n_cols

        fig = make_subplots(
            rows=n_rows,
            cols=n_cols,
            subplot_titles=[f"{s}" for s in symbols],
            vertical_spacing=0.10,
            horizontal_spacing=0.08,
        )

        for i, sym in enumerate(symbols):
            r = i // n_cols + 1
            c = i % n_cols + 1

            symbol_df = plot_df[plot_df["symbol"] == sym]
            max_val = symbol_df["abs_profit"].max()
            bin_size = max_val / 30 if max_val > 0 else 1

            fig.add_trace(
                go.Histogram(
                    x=symbol_df[symbol_df["result"] == "win"]["abs_profit"],
                    name="Wins",
                    marker_color="green",
                    opacity=0.75,
                    xbins=dict(start=0, end=max_val, size=bin_size),
                    legendgroup="wins",
                    showlegend=(i == 0),
                ),
                row=r,
                col=c,
            )
            fig.add_trace(
                go.Histogram(
                    x=symbol_df[symbol_df["result"] == "loss"]["abs_profit"],
                    name="Losses",
                    marker_color="red",
                    opacity=0.75,
                    xbins=dict(start=0, end=max_val, size=bin_size),
                    legendgroup="losses",
                    showlegend=(i == 0),
                ),
                row=r,
                col=c,
            )

            fig.update_xaxes(title_text="Absolute Profit", row=r, col=c)
            fig.update_yaxes(title_text="Count", row=r, col=c)

        fig.update_layout(
            title="Distribution of absolute profit for closed trades by symbol",
            barmode="overlay",
            template="plotly_white",
            height=max(400, 320 * n_rows),
        )
        fig.show()

## Best and worst closed trades

In [8]:
if closed_trades.empty:
    print("No closed trades available for ranking.")
else:
    cols = ["time", "symbol", "side", "entry_type", "volume", "price", "profit", "commission", "swap", "fee", "net_pnl", "comment"]
    print("Top 10 winners")
    display(closed_trades.sort_values("net_pnl", ascending=False)[cols].head(10))
    print("Top 10 losers")
    display(closed_trades.sort_values("net_pnl", ascending=True)[cols].head(10))

Top 10 winners


,time,symbol,side,entry_type,volume,price,profit,commission,swap,fee,net_pnl,comment
125,2026-04-23 16:40:52+00:00,XAUUSD.a,sell,OUT,0.10,4735.10000,102.13,0.00,0.0,0.0,102.13,[tp 4735.10]
123,2026-04-23 15:06:56+00:00,EURUSD.a,sell,OUT,1.25,1.16953,104.85,-4.38,0.0,0.0,100.47,[tp 1.16953]
27,2026-04-21 18:26:30+00:00,EURUSD.a,buy,OUT,1.25,1.17514,102.93,-4.38,0.0,0.0,98.55,[tp 1.17514]
155,2026-04-24 14:21:45+00:00,EURUSD.a,sell,OUT,1.25,1.17116,99.74,-4.38,0.0,0.0,95.36,[tp 1.17116]
39,2026-04-22 02:13:35+00:00,XAUUSD.a,sell,OUT,0.10,4727.05000,91.02,0.00,0.0,0.0,91.02,[tp 4727.05]
77,2026-04-22 18:17:39+00:00,XAUUSD.a,buy,OUT,0.10,4731.14000,89.96,0.00,0.0,0.0,89.96,[tp 4731.14]
38,2026-04-22 02:12:53+00:00,XAUUSD.a,sell,OUT,0.10,4726.69000,87.53,0.00,0.0,0.0,87.53,[tp 4726.69]
129,2026-04-23 18:19:27+00:00,EURUSD.a,sell,OUT,1.25,1.17124,89.02,-4.38,0.0,0.0,84.64,[tp 1.17124]
143,2026-04-24 05:10:12+00:00,XAUUSD.a,buy,OUT,0.10,4692.60000,83.07,0.00,0.0,0.0,83.07,[tp 4692.60]
180,2026-04-24 19:52:00+00:00,EURUSD.a,sell,OUT,1.25,1.17197,87.40,-4.38,0.0,0.0,83.02,[tp 1.17197]


Top 10 losers


,time,symbol,side,entry_type,volume,price,profit,commission,swap,fee,net_pnl,comment
170,2026-04-24 15:45:47+00:00,XAUUSD.a,sell,OUT,0.10,4697.5400,-143.08,0.00,0.0,0.0,-143.08,[sl 4697.54]
169,2026-04-24 15:44:33+00:00,XAUUSD.a,sell,OUT,0.10,4698.7400,-136.77,0.00,0.0,0.0,-136.77,[sl 4698.74]
23,2026-04-21 17:32:42+00:00,XAUUSD.a,sell,OUT,0.10,4769.3800,-130.10,0.00,0.0,0.0,-130.10,[sl 4769.38]
168,2026-04-24 15:37:31+00:00,EURUSD.a,sell,OUT,1.25,1.1707,-112.09,-4.38,0.0,0.0,-116.47,[sl 1.17070]
171,2026-04-24 15:46:43+00:00,XAUUSD.a,sell,OUT,0.10,4697.0900,-106.49,0.00,0.0,0.0,-106.49,[sl 4697.09]
74,2026-04-22 17:04:53+00:00,XAUUSD.a,sell,OUT,0.10,4745.4700,-104.39,0.00,0.0,0.0,-104.39,[sl 4745.47]
177,2026-04-24 18:34:56+00:00,XAUUSD.a,sell,OUT,0.10,4718.4900,-101.99,0.00,0.0,0.0,-101.99,[sl 4718.49]
101,2026-04-23 04:13:09+00:00,XAUUSD.a,sell,OUT,0.10,4737.3700,-101.28,0.00,0.0,0.0,-101.28,[sl 4737.37]
106,2026-04-23 04:23:27+00:00,XAUUSD.a,sell,OUT,0.10,4728.3500,-100.90,0.00,0.0,0.0,-100.90,[sl 4728.35]
108,2026-04-23 04:29:44+00:00,XAUUSD.a,sell,OUT,0.10,4727.7500,-99.10,0.00,0.0,0.0,-99.10,[sl 4727.75]
